# 🌍 PetroMind — Module 1: Porosity Prediction (Supervised Regression)

**Objective:** Predict core porosity (PHIE) from wireline logs using ML.

**Why this problem?** Core porosity is expensive to measure but cheap to estimate from logs.
A good regression model saves millions in coring costs.

| Feature | Description |
|---------|-------------|
| GR | Gamma Ray (API) — shaliness indicator |
| RHOB | Bulk Density (g/cc) — directly tied to porosity via physics |
| NPHI | Neutron Porosity (v/v) — sensitive to hydrogen content |
| DT | Sonic Slowness (µs/ft) — Wyllie time-average relationship |
| RT | Resistivity (log₁₀, ohm·m) — fluid type indicator |

**Target:** PHIE — effective porosity (0–0.38 v/v)


## ⚙️ Setup


In [ ]:
# ── Standard notebook setup ───────────────────────────────────────────────────
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # safe for all OS; change to "inline" if plots don't show
import matplotlib.pyplot as plt
%matplotlib inline             # comment out if not in Jupyter
warnings.filterwarnings("ignore")
np.random.seed(42)

# Project root (works whether you run from /notebooks or the project root)
ROOT   = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
OUTDIR = os.path.join(ROOT, "outputs")
DATDIR = os.path.join(ROOT, "data")
os.makedirs(OUTDIR, exist_ok=True)
os.makedirs(DATDIR, exist_ok=True)
print(f"ROOT   = {ROOT}")
print(f"OUTDIR = {OUTDIR}")
print(f"DATDIR = {DATDIR}")


## 📊 Step 1 — Generate Synthetic Well Log Data

We mimic realistic petrophysical relationships:
- **Sand** → low RHOB, low NPHI, low GR → **HIGH porosity**
- **Shale** → high RHOB, high NPHI, high GR → **LOW porosity**

Compaction trend + realistic noise are added.
5 wells are generated to allow proper well-based splitting.


In [ ]:
def generate_well_logs(n_samples=2000, n_wells=5):
    records = []
    rma, rfl   = 2.65, 1.05
    dtma, dtfl = 55.5, 189.0

    for wid in range(n_wells):
        n     = n_samples // n_wells
        depth = np.linspace(2000 + wid*50, 3500 + wid*50, n)
        depth_norm = (depth - depth.min()) / (depth.max() - depth.min())
        compaction = 1.0 - 0.30 * depth_norm

        blocks = np.random.choice([0,1,2,3], size=40, p=[0.35,0.25,0.20,0.20])
        facies = np.repeat(blocks, n//40 + 1)[:n]

        base_phi = {0:0.05, 1:0.10, 2:0.18, 3:0.28}
        phi = np.array([base_phi[f] for f in facies])
        phi = np.clip(phi * compaction + np.random.normal(0, 0.015, n), 0.01, 0.38)

        gr_base = {0:110, 1:75, 2:50, 3:22}
        gr   = np.clip(np.array([gr_base[f] for f in facies]) + np.random.normal(0,8,n), 10, 165)
        rhob = np.clip((1-phi)*rma + phi*rfl + np.random.normal(0,0.03,n), 2.0, 2.85)

        vsh  = np.array([f/3.0 for f in facies])
        nphi = np.clip(phi + 0.15*vsh + np.random.normal(0,0.02,n), 0.01, 0.55)
        dt   = np.clip(1/((1-phi)/dtma + phi/dtfl) + np.random.normal(0,2.5,n), 50, 140)

        rt_base = {0:1.2, 1:2.0, 2:5.0, 3:15.0}
        rt = np.clip(np.array([rt_base[f] for f in facies])*np.random.lognormal(0,0.4,n), 0.5, 200)

        dphi   = (rma - rhob) / (rma - rfl)
        xphi   = (nphi + dphi) / 2.0
        vsh_gr = np.clip((gr - 15) / 150, 0, 1)

        for i in range(n):
            records.append({
                "well_id": wid, "depth": depth[i], "facies": facies[i],
                "GR": gr[i], "RHOB": rhob[i], "NPHI": nphi[i],
                "DT": dt[i], "RT": np.log10(rt[i]),
                "DPHI": dphi[i], "XPHI": xphi[i], "VSH_GR": vsh_gr[i],
                "PHIE": phi[i],
            })
    return pd.DataFrame(records)

df = generate_well_logs(n_samples=2000, n_wells=5)
print(f"Dataset shape: {df.shape}")
df[["GR","RHOB","NPHI","DT","RT","PHIE"]].describe().round(3)


## 🔍 Step 2 — Exploratory Data Analysis

> **Geoscience rule:** Always plot logs before modelling!
Look for depth trends, outliers, and physics-based correlations.


In [ ]:
import seaborn as sns

# ── Well log strip ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 6, figsize=(14, 10), sharey=True)
logs    = ["GR","RHOB","NPHI","DT","RT","PHIE"]
colors  = ["green","red","blue","saddlebrown","black","purple"]
xlabels = ["GR (API)","RHOB (g/cc)","NPHI (v/v)","DT (µs/ft)","log₁₀(RT)","PHIE (v/v)"]

well0 = df[df["well_id"]==0].reset_index(drop=True)
for ax, log, col, xl in zip(axes, logs, colors, xlabels):
    ax.plot(well0[log], well0["depth"], color=col, lw=0.5)
    ax.set_xlabel(xl, fontsize=8); ax.invert_yaxis(); ax.grid(alpha=0.3)
axes[0].set_ylabel("Depth (m)")
fig.suptitle("Well 0 — Log Suite", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR,"M1_logs.png"), dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────────
feat_cols = ["GR","RHOB","NPHI","DT","RT","DPHI","XPHI","VSH_GR","PHIE"]
fig, ax = plt.subplots(figsize=(9,7))
sns.heatmap(df[feat_cols].corr(), annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, ax=ax, annot_kws={"size":8})
ax.set_title("Feature Correlation Matrix", fontweight="bold")
plt.tight_layout(); plt.savefig(os.path.join(OUTDIR,"M1_correlation.png"), dpi=120, bbox_inches="tight")
plt.show()


## ✂️ Step 3 — Train / Val / Test Split (**Well-Based!**)

> ⚠️ **Critical Pitfall — Depth Correlation Leakage:**  
> Randomly splitting rows puts adjacent depth samples (nearly identical) in both
> train and test → artificially inflated R².  
> **Fix:** Split by **Well ID** (spatial cross-validation).

- **Train:** Wells 0, 1, 2
- **Val:**   Well 3
- **Test:**  Well 4 ← held out, never seen


In [ ]:
from sklearn.preprocessing import StandardScaler

FEATURES = ["GR","RHOB","NPHI","DT","RT","DPHI","XPHI","VSH_GR"]
TARGET   = "PHIE"

train_df = df[df["well_id"].isin([0,1,2])]
val_df   = df[df["well_id"]==3]
test_df  = df[df["well_id"]==4]

X_train, y_train = train_df[FEATURES].values, train_df[TARGET].values
X_val,   y_val   = val_df[FEATURES].values,   val_df[TARGET].values
X_test,  y_test  = test_df[FEATURES].values,  test_df[TARGET].values

# Fit scaler ONLY on training data!
scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)
print(f"Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}")


## 📏 Step 4 — Baseline Model

> Always define a baseline first. If your ML model can't beat
> 'predict the mean', something is wrong.


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

baseline_pred = np.full_like(y_test, y_train.mean())
baseline_mae  = mean_absolute_error(y_test, baseline_pred)
print(f"Baseline (predict-mean) MAE: {baseline_mae:.4f}  ({baseline_mae*100:.2f} porosity units)")


## 🤖 Step 5 — Train Models (Ridge → RF → GBM → XGBoost)


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

def evaluate(model, X_tr, y_tr, X_te, y_te, name):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    mae  = mean_absolute_error(y_te, pred)
    rmse = np.sqrt(mean_squared_error(y_te, pred))
    r2   = r2_score(y_te, pred)
    print(f"  {name:<35s}  MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}")
    return model, pred

print("--- Validation Results ---")
m1, _ = evaluate(Ridge(alpha=1.0), X_train_s, y_train, X_val_s, y_val, "Ridge Regression")
m2, _ = evaluate(RandomForestRegressor(n_estimators=200, max_depth=12, n_jobs=-1, random_state=42),
                 X_train_s, y_train, X_val_s, y_val, "Random Forest")
m3, _ = evaluate(GradientBoostingRegressor(n_estimators=300, max_depth=5, learning_rate=0.05,
                 subsample=0.8, random_state=42),
                 X_train_s, y_train, X_val_s, y_val, "Gradient Boosting")

try:
    from xgboost import XGBRegressor
    m4, _ = evaluate(XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.05,
                     subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0),
                     X_train_s, y_train, X_val_s, y_val, "XGBoost")
    best_model = m4; print("  → Best: XGBoost")
except ImportError:
    best_model = m3; print("  → Best: Gradient Boosting")


In [ ]:
print("--- FINAL TEST RESULTS (Well 4 — never seen) ---")
test_pred = best_model.predict(X_test_s)
test_mae  = mean_absolute_error(y_test, test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
test_r2   = r2_score(y_test, test_pred)
print(f"  MAE  = {test_mae:.4f}  ({test_mae*100:.2f} porosity units)")
print(f"  RMSE = {test_rmse:.4f}")
print(f"  R²   = {test_r2:.4f}")
print(f"  Improvement over baseline: {(baseline_mae-test_mae)/baseline_mae*100:.1f} %")


## 🔍 Step 6 — Feature Importance & Explainability

> **Permutation importance** (model-agnostic) tells us which features
> the model actually relies on. If NPHI is top but you're in a gas zone
> where NPHI is unreliable → investigate before trusting the model!


In [ ]:
from sklearn.inspection import permutation_importance

perm   = permutation_importance(best_model, X_val_s, y_val, n_repeats=10, random_state=42)
imp_df = pd.DataFrame({"feature": FEATURES,
                        "importance": perm.importances_mean,
                        "std": perm.importances_std}).sort_values("importance", ascending=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].barh(imp_df["feature"], imp_df["importance"], xerr=imp_df["std"],
             color="steelblue", alpha=0.8)
axes[0].set_title("Permutation Feature Importance", fontweight="bold")
axes[0].set_xlabel("Mean decrease in R²")

axes[1].scatter(y_test, test_pred, alpha=0.3, s=5, c="steelblue")
lims = [min(y_test.min(),test_pred.min()), max(y_test.max(),test_pred.max())]
axes[1].plot(lims, lims, "r--", lw=2, label="Perfect"); axes[1].legend()
axes[1].set_xlabel("True PHIE (v/v)"); axes[1].set_ylabel("Predicted PHIE (v/v)")
axes[1].set_title(f"Test: Predicted vs Actual  R²={test_r2:.3f}", fontweight="bold")

depth_te = test_df["depth"].values; si = np.argsort(depth_te)
axes[2].plot(y_test[si],    depth_te[si], "b-",  lw=1, label="True",      alpha=0.8)
axes[2].plot(test_pred[si], depth_te[si], "r--", lw=1, label="Predicted", alpha=0.8)
axes[2].invert_yaxis(); axes[2].grid(alpha=0.3)
axes[2].set_xlabel("PHIE (v/v)"); axes[2].set_ylabel("Depth (m)")
axes[2].set_title("Test Well Log: True vs Predicted", fontweight="bold"); axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTDIR,"M1_results.png"), dpi=120, bbox_inches="tight")
plt.show()


## 📉 Step 7 — Uncertainty Estimation (P10–P90 Quantile Regression)

> Knowing 'porosity is **0.18–0.24 with 80% confidence**' is far more
> valuable than a single point estimate in reservoir modelling.


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor as GBR

gb_low  = GBR(loss="quantile", alpha=0.10, n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
gb_high = GBR(loss="quantile", alpha=0.90, n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
gb_low.fit(X_train_s, y_train); gb_high.fit(X_train_s, y_train)

pred_low  = gb_low.predict(X_test_s)
pred_high = gb_high.predict(X_test_s)
coverage  = np.mean((y_test >= pred_low) & (y_test <= pred_high))
print(f"P10–P90 interval coverage: {coverage*100:.1f} %  (target ≈ 80 %)")


## 💾 Step 8 — Save Data for Next Modules


In [ ]:
df["PHIE_pred"] = best_model.predict(scaler.transform(df[FEATURES].values))
df.to_csv(os.path.join(DATDIR,"petromind_logs.csv"), index=False)
print(f"Saved → data/petromind_logs.csv  ({len(df)} rows)")


## ✅ Module 1 Complete

**What to try next:**
- Hyperparameter tuning with Optuna
- Gaussian Process Regression for full uncertainty
- SHAP values for per-sample explanations
- Replace synthetic data with your own LAS files (`pip install lasio`)

**Common Pitfalls:**
1. 🚨 Depth leakage — never random-split depth samples
2. 🚨 Always use `log10(RT)` — raw resistivity spans 4 orders of magnitude
3. 🚨 Fit `StandardScaler` only on `X_train`
